# Model Inference

Loads the 5 trained models from MLflow, constructs a feature vector for a new match, and returns predictions for all targets.

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import pickle
from pathlib import Path

DATA_PATH  = Path('../../datasets/processed/feature_engineered_dataset.csv')
MODELS_DIR = Path('../../models')

mlflow.set_tracking_uri('file:../../mlruns')
mlflow.set_registry_uri('sqlite:///../../mlflow.db')

FEATURE_COLS = [
    # categorical (htr_encoded excluded: it is the current match HT result = leakage)
    "home_team_encoded", "away_team_encoded", "referee_encoded",
    "day_encoded", "month_sin", "month_cos",
    # team form
    "home_points_last5", "home_goals_scored_avg5", "home_goals_conceded_avg5",
    "home_sot_avg5", "home_clean_sheets_last5",
    "home_points_last10", "home_goals_scored_avg10", "home_goals_conceded_avg10",
    "home_win_streak",
    "away_points_last5", "away_goals_scored_avg5", "away_goals_conceded_avg5",
    "away_sot_avg5", "away_clean_sheets_last5",
    "away_points_last10", "away_goals_scored_avg10", "away_goals_conceded_avg10",
    "away_win_streak",
    # home/away split
    "home_win_rate_last10", "home_goals_avg_last10",
    "away_win_rate_last10", "away_goals_avg_last10",
    # head-to-head
    "h2h_meetings", "h2h_home_win_rate", "h2h_avg_total_goals", "h2h_btts_rate",
    # shot quality
    "home_conversion_rate_avg5", "home_sot_ratio_avg5",
    "away_conversion_rate_avg5", "away_sot_ratio_avg5",
    # half-time patterns (rolling from previous matches, not the current one)
    "home_2nd_half_goals_avg5", "home_lead_hold_rate", "home_comeback_rate",
    "away_2nd_half_goals_avg5", "away_lead_hold_rate", "away_comeback_rate",
    # referee
    "ref_avg_yellows", "ref_avg_fouls", "ref_home_win_rate",
    # temporal
    "match_week", "season_phase", "home_days_rest", "away_days_rest",
]


## 1. Load Models and Encoders from MLflow

In [ ]:
def load_model(name):
    uri = f'models:/{name}/latest'
    return mlflow.xgboost.load_model(uri)

model_result  = load_model('soca_result')
model_btts    = load_model('soca_btts')
model_over25  = load_model('soca_over25')
model_over15  = load_model('soca_over15')
model_goals   = load_model('soca_goals')

with open(MODELS_DIR / 'team_encoder.pkl', 'rb') as f:
    team_encoder = pickle.load(f)
with open(MODELS_DIR / 'referee_encoder.pkl', 'rb') as f:
    referee_encoder = pickle.load(f)

df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print('All models and encoders loaded.')

## 2. Feature Lookup

For a new match, we look up the most recent row for each team in the historical dataset to get their latest form features. H2H is computed from all prior meetings between the two teams.

In [ ]:
def get_latest_team_features(team, side, df):
    col = 'HomeTeam' if side == 'home' else 'AwayTeam'
    rows = df[df[col] == team]
    if rows.empty:
        raise ValueError(f'Team not found: {team}')
    latest = rows.iloc[-1]
    prefix = side + '_'
    cols   = [c for c in FEATURE_COLS if c.startswith(prefix)]
    return {c: latest[c] for c in cols}

def get_h2h_features(home_team, away_team, df):
    h2h = df[
        ((df['HomeTeam'] == home_team) & (df['AwayTeam'] == away_team)) |
        ((df['HomeTeam'] == away_team) & (df['AwayTeam'] == home_team))
    ]
    if h2h.empty:
        return {'h2h_meetings': 0, 'h2h_home_win_rate': 0.33,
                'h2h_avg_total_goals': 2.5, 'h2h_btts_rate': 0.5}
    latest = h2h.iloc[-1]
    return {c: latest[c] for c in ['h2h_meetings', 'h2h_home_win_rate',
                                    'h2h_avg_total_goals', 'h2h_btts_rate']}

def get_referee_features(referee, df):
    rows = df[df['Referee'] == referee]
    if rows.empty:
        return {'ref_avg_yellows': df['ref_avg_yellows'].mean(),
                'ref_avg_fouls': df['ref_avg_fouls'].mean(),
                'ref_home_win_rate': df['ref_home_win_rate'].mean()}
    latest = rows.iloc[-1]
    return {c: latest[c] for c in ['ref_avg_yellows', 'ref_avg_fouls', 'ref_home_win_rate']}

print('Feature lookup helpers defined.')

## 3. Build Feature Vector for a New Match

In [ ]:
# --- Change these to predict any match ---
HOME_TEAM = 'Arsenal'
AWAY_TEAM = 'Chelsea'
DATE      = '2025-08-16'
REFEREE   = 'Michael Oliver'
MATCH_WEEK = 1

date_obj   = pd.to_datetime(DATE)
day_order  = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
              'Friday': 4, 'Saturday': 5, 'Sunday': 6}
month_order = {'January': 1, 'February': 2, 'March': 3, 'April': 4,
               'May': 5, 'June': 6, 'July': 7, 'August': 8,
               'September': 9, 'October': 10, 'November': 11, 'December': 12}

month_num  = date_obj.month
day_name   = date_obj.day_name()

features = {}
features['home_team_encoded'] = int(team_encoder.transform([HOME_TEAM])[0])
features['away_team_encoded'] = int(team_encoder.transform([AWAY_TEAM])[0])
features['referee_encoded']   = int(referee_encoder.transform([REFEREE])[0])
features['day_encoded']        = day_order.get(day_name, 5)
features['month_sin']          = np.sin(2 * np.pi * month_num / 12)
features['month_cos']          = np.cos(2 * np.pi * month_num / 12)

features.update(get_latest_team_features(HOME_TEAM, 'home', df))
features.update(get_latest_team_features(AWAY_TEAM, 'away', df))
features.update(get_h2h_features(HOME_TEAM, AWAY_TEAM, df))
features.update(get_referee_features(REFEREE, df))

features['match_week']    = MATCH_WEEK
features['season_phase']  = 0 if MATCH_WEEK <= 10 else (1 if MATCH_WEEK <= 28 else 2)
features['home_days_rest'] = 7.0
features['away_days_rest'] = 7.0

X = pd.DataFrame([features])[FEATURE_COLS]
print(f'Feature vector shape: {X.shape}')
X

## 4. Run Predictions

In [ ]:
result_proba = model_result.predict_proba(X)[0]   # [away, draw, home]
result_pred  = model_result.predict(X)[0]

btts_proba   = model_btts.predict_proba(X)[0][1]
over25_proba = model_over25.predict_proba(X)[0][1]
over15_proba = model_over15.predict_proba(X)[0][1]
goals_pred   = model_goals.predict(X)[0]

result_map   = {0: 'Away Win', 1: 'Draw', 2: 'Home Win'}

print(f'Match: {HOME_TEAM} vs {AWAY_TEAM} on {DATE}')
print()
print(f'Result         : {result_map[result_pred]}')
print(f'  Home Win prob : {result_proba[2]:.1%}')
print(f'  Draw prob     : {result_proba[1]:.1%}')
print(f'  Away Win prob : {result_proba[0]:.1%}')
print()
print(f'BTTS           : {"Yes" if btts_proba > 0.5 else "No"} ({btts_proba:.1%})')
print(f'Over 2.5 goals : {"Yes" if over25_proba > 0.5 else "No"} ({over25_proba:.1%})')
print(f'Over 1.5 goals : {"Yes" if over15_proba > 0.5 else "No"} ({over15_proba:.1%})')
print(f'Predicted goals: {goals_pred:.1f}')